# Thesis Data Science & Synthesis Notebook

This notebook serves as the core analytical pipeline for the Bachelor's Thesis on **Swarm-Based Geometric Transport**.
It processes trial telemetry, evaluates success probabilities across swarm densities, and quantifies the "Torque Debt" induced by non-convex payload geometry.

## 1. Data Ingestion
Robust function to crawl the `/logs` directory, parse all JSON trial files, and aggregate them into a single Pandas DataFrame.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set publication-quality aesthetics
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)

def ingest_telemetry_data(base_dir='logs'):
    """
    Crawls the specified log directory to parse JSON trial files.
    Returns a comprehensive Pandas DataFrame.
    """
    all_data = []
    if not os.path.exists(base_dir):
        print(f"Warning: Directory '{base_dir}' not found.")
        return pd.DataFrame()
        
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.json'):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, 'r') as f:
                        data = json.load(f)
                        
                        # Extract shape from folder name if needed
                        folder_name = os.path.basename(root)
                        if 'shape' not in data:
                            data['shape'] = folder_name
                        
                        all_data.append(data)
                except Exception as e:
                    print(f"Failed to parse {file_path}: {e}")
                    
    df = pd.DataFrame(all_data)
    return df

df = ingest_telemetry_data()
df.head()

## 2. Success Rate Analysis
Evaluating the Success % per Swarm Size ($N$). This demonstrates the carrying capacity and the "Clogging" threshold where physical interference supersedes recruitment benefits.

In [ ]:
if not df.empty and 'N' in df.columns:
    success_rates = df.groupby('N')['success'].mean() * 100
    
    plt.figure(figsize=(8, 5))
    sns.barplot(x=success_rates.index, y=success_rates.values, palette='viridis')
    plt.title('Success Rate vs. Swarm Size ($N$)', fontweight='bold')
    plt.xlabel('Swarm Size ($N$)')
    plt.ylabel('Success Rate (%)')
    plt.axhline(50, color='r', linestyle='--', label='50% Threshold')
    plt.legend()
    plt.tight_layout()
    plt.show()

**Scientific Interpretation of the "Clogging" Threshold:**
The success rate reveals a non-linear relationship with swarm size. Initially, an increase in $N$ provides necessary force to overcome static friction and payload inertia, increasing success probability. However, as $N$ surpasses a critical density threshold, agents begin to physically impede one another. This "clogging" effect generates conflicting force vectors and prevents optimal spatial distribution around the payload, resulting in a precipitous drop in the success rate. It serves as empirical evidence that over-recruitment is detrimental to collective transport in confined spaces.

## 3. Efficiency Distributions
Visualizing 'Time to Success' vs. $N$ and the correlation between 'Total Shuffles' and 'Path Tortuosity'.

In [ ]:
if not df.empty and 'N' in df.columns and 'duration' in df.columns:
    successful_trials = df[df['success'] == True]
    
    plt.figure(figsize=(10, 5))
    sns.violinplot(data=successful_trials, x='N', y='duration', palette='mako', inner="quartile")
    plt.title('Distribution of Time to Success across Swarm Sizes', fontweight='bold')
    plt.xlabel('Swarm Size ($N$)')
    plt.ylabel('Duration (s)')
    plt.tight_layout()
    plt.show()

In [ ]:
if not df.empty and 'total_shuffles' in df.columns and 'tortuosity' in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x='total_shuffles', y='tortuosity', hue='success', palette='Set1', alpha=0.7, s=80)
    plt.title('Path Tortuosity vs. Total Swarm Shuffles', fontweight='bold')
    plt.xlabel('Total Shuffles')
    plt.ylabel('Path Tortuosity')
    plt.tight_layout()
    plt.show()

## 4. The "Torque Debt" Section (Geometry Comparison)
A direct comparison of Square vs. L-Shape performance to prove the necessity of the shuffling behavior.

In [ ]:
if not df.empty and 'shape' in df.columns and 'total_shuffles' in df.columns:
    print("=== Mean & Standard Deviation of Shuffles by Shape ===")
    comp_table = df.groupby('shape')['total_shuffles'].agg(['mean', 'std', 'count']).round(2)
    display(comp_table)
    
    # T-Test for Statistical Significance
    square_shuffles = df[df['shape'] == 'square']['total_shuffles'].dropna()
    l_shape_shuffles = df[df['shape'] == 'l_shape']['total_shuffles'].dropna()
    
    if len(square_shuffles) > 1 and len(l_shape_shuffles) > 1:
        t_stat, p_val = stats.ttest_ind(square_shuffles, l_shape_shuffles, equal_var=False)
        print(f"\nWelch's t-test Results:")
        print(f"T-Statistic: {t_stat:.4f}")
        print(f"P-Value: {p_val:.4e}")
        if p_val < 0.05:
            print("Conclusion: The difference in total shuffles between the Square and L-Shape is statistically significant (p < 0.05).")
        else:
            print("Conclusion: No statistically significant difference in shuffles between shapes with current sample size.")
    else:
        print("\nInsufficient data samples to perform t-test.")

## 5. Behavioral Dynamics
Correlation heatmap mapping the relationships between continuous operational variables.

In [ ]:
if not df.empty:
    cols = ['duration', 'total_shuffles', 'tortuosity', 'success']
    avail_cols = [c for c in cols if c in df.columns]
    
    if avail_cols:
        corr_df = df[avail_cols].copy()
        if 'success' in corr_df.columns:
            corr_df['success'] = corr_df['success'].astype(float)
            
        corr_matrix = corr_df.corr()
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
        plt.title('Correlation Matrix: Behavioral Dynamics', fontweight='bold')
        plt.tight_layout()
        plt.show()